In [1]:
# Step 1: Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
from sklearn.model_selection import ParameterGrid
import tensorflow as tf
from sklearn.model_selection import GridSearchCV
import numpy as np
from matplotlib import pyplot as plt
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score, roc_auc_score, roc_curve

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install scikeras





In [3]:
from scikeras.wrappers import KerasClassifier


In [4]:
# Set global seeds for reproducibility
seed = 123
tf.random.set_seed(seed)
np.random.seed(seed)

In [5]:
data_dir = '/content/drive/My Drive/breakhis'  # Replace with your actual path to the dataset


In [6]:
# Define the model-building function with hyperparameters
def build_model(learning_rate=1e-3):
    model = tf.keras.Sequential()
    model.add(tf.keras.layers.Conv2D(32, (3, 3), activation='relu', input_shape=(256, 256, 3)))
    model.add(tf.keras.layers.MaxPooling2D((2, 2)))
    model.add(tf.keras.layers.Conv2D(64, (3, 3), activation='relu'))
    model.add(tf.keras.layers.MaxPooling2D((2, 2)))
    model.add(tf.keras.layers.Conv2D(128, (3, 3), activation='relu'))
    model.add(tf.keras.layers.MaxPooling2D((2, 2)))
    model.add(tf.keras.layers.Dropout(0.4))
    model.add(tf.keras.layers.Flatten())
    model.add(tf.keras.layers.Dense(256, activation='relu'))
    model.add(tf.keras.layers.Dense(128, activation='relu'))
    model.add(tf.keras.layers.Dense(1, activation='sigmoid'))  # Binary classification output

    # Hyperparameter for learning rate
    #learning_rate = hp.Choice('learning_rate', values=[1e-4, 2e-4, 3e-4])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=learning_rate),
                  loss='binary_crossentropy',
                  metrics=['accuracy'])
    return model


In [7]:
# Step 3: Load and prepare datasets with a given batch size
img_size = (256, 256)

def prepare_datasets(batch_size):
    train_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="training",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    val_dataset = tf.keras.utils.image_dataset_from_directory(
        data_dir,
        validation_split=0.2,
        subset="validation",
        seed=seed,
        image_size=img_size,
        batch_size=batch_size,
        shuffle=True
    )

    # Data Augmentation and Normalization
    data_augmentation = tf.keras.Sequential([
        tf.keras.layers.RandomFlip("horizontal_and_vertical"),
        tf.keras.layers.RandomRotation(0.2),
        tf.keras.layers.RandomZoom(0.2),
        tf.keras.layers.RandomContrast(0.2),
        tf.keras.layers.Lambda(lambda x: tf.image.random_brightness(x, max_delta=0.3)),
        tf.keras.layers.Lambda(lambda x: tf.image.random_saturation(x, lower=0.8, upper=1.2)),
    ])

    normalization_layer = tf.keras.layers.Rescaling(1./255)
    train_dataset = train_dataset.map(lambda x, y: (data_augmentation(normalization_layer(x), training=True), y))
    val_dataset = val_dataset.map(lambda x, y: (normalization_layer(x), y))

    return train_dataset, val_dataset

In [8]:
# Step 4: Define the Keras Tuner for hyperparameter search
#tuner = kt.Hyperband(
    #build_model,
    #objective='val_accuracy',
    #max_epochs=10,
    #factor=3,
    #directory='kt_tuning',
    #project_name='cnn_tune'
#)

In [9]:
# Wrap model using KerasClassifier
model = KerasClassifier(build_fn=build_model, verbose=1)

In [10]:
# Define GridSearch parameter grid
param_grid = {
    'learning_rate': [1e-4, 2e-4, 3e-4 ,4e-4],
    'batch_size': [16, 32, 64],
    'epochs': [10, 20]
}

In [11]:
grid = GridSearchCV(estimator=model, param_grid=param_grid, cv=3)


In [12]:
# Load datasets and run grid search
best_model = None
train_dataset, val_dataset = prepare_datasets(batch_size=32)

Found 7670 files belonging to 2 classes.
Using 6136 files for training.
Found 7670 files belonging to 2 classes.
Using 1534 files for validation.


In [ ]:
grid_result = grid.fit(
    np.concatenate([x for x, _ in train_dataset]),
    np.concatenate([y for _, y in train_dataset])
)

In [ ]:
# Print all results
print("Grid Search Results:")
for params, mean_score, std_score in zip(grid_result.cv_results_['params'],
                                         grid_result.cv_results_['mean_test_score'],
                                         grid_result.cv_results_['std_test_score']):
    print(f"Params: {params} | Mean Accuracy: {mean_score:.4f} | Std: {std_score:.4f}")

In [ ]:
# Print best results
print(f"Best Parameters: {grid_result.best_params_}")
print(f"Best Score: {grid_result.best_score_}")

In [ ]:
# Evaluate best model
best_model = grid_result.best_estimator_.model
best_batch_size = grid_result.best_params_['batch_size']

In [ ]:
# Load test data
test_dataset = tf.keras.utils.image_dataset_from_directory(
    data_dir,
    validation_split=0.2,
    subset="validation",
    seed=seed,
    image_size=img_size,
    batch_size=best_batch_size
)

In [ ]:
test_dataset = test_dataset.map(lambda x, y: (normalization_layer(x), y))



In [ ]:
y_true = []
y_pred_probs = []
for batch in test_dataset.as_numpy_iterator():
    X, y = batch
    y_pred = best_model.predict(X)
    y_true.extend(y)
    y_pred_probs.extend(y_pred)

In [ ]:
# Convert lists to numpy arrays
y_true = np.array(y_true)
y_pred_probs = np.array(y_pred_probs)

In [ ]:
# Evaluation metrics
y_pred_binary = (y_pred_probs > 0.5).astype(int)
print(f'Accuracy: {accuracy_score(y_true, y_pred_binary)}')
print(f'Precision: {precision_score(y_true, y_pred_binary)}')
print(f'Recall: {recall_score(y_true, y_pred_binary)}')
print(f'F1 Score: {f1_score(y_true, y_pred_binary)}')
print(f'ROC AUC: {roc_auc_score(y_true, y_pred_probs)}')

In [ ]:
# Save the best model
import os
output_dir = '/content/drive/My Drive/saved_model'
os.makedirs(output_dir, exist_ok=True)
best_model.save(os.path.join(output_dir, 'cnn_model_gridsearch.keras'))
print(f"Model saved at: {output_dir}")